In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE       = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2")
MASTER_CSV = BASE / "03_processed" / "master_ml.csv"

df = pd.read_csv(MASTER_CSV, sep=";", low_memory=False)

REGROUPEMENT = {
    "Extreme gauche"      : "Gauche",
    "Gauche radicale"     : "Gauche",
    "Gauche"              : "Gauche",
    "Ecologie"            : "Gauche",
    "Centre"              : "Centre",
    "Droite"              : "Droite",
    "Droite souverainiste": "Droite",
    "Extreme droite"      : "Extreme droite",
    "Divers"              : "Divers",
    "Inconnu"             : "Divers",
}
df["famille_regroupee"] = df["famille_gagnante"].map(REGROUPEMENT).fillna("Divers")

COLS_SOCIO = [
    "population", "densite", "superficie_km2",
    "nb_logements", "nb_residences_principales", "nb_logements_vacants",
    "actifs_2022", "chomeurs_2022", "actifs_occupes_2022", "inactifs_2022",
    "taux_chomage_2022", "revenu_median", "taux_pauvrete",
    "part_chomeurs", "part_foyers_imposes",
    "d1_revenu", "d9_revenu", "ratio_interdecile",
    "nb_immigres", "nb_non_immigres", "total_delits",
    "nb_menages", "nb_personnes_menages"
]
COLS_EURO = [
    "euro_centre_ratio", "euro_divers_ratio", "euro_droite_ratio",
    "euro_extreme_droite_ratio", "euro_extreme_gauche_ratio",
    "euro_gauche_ratio", "euro_gauche_radicale_ratio",
    "euro_droite_souverainiste_ratio", "euro_ecologie_ratio",
    "euro_taux_participation"
]

# ==============================================================================
# 1. STRUCTURE GENERALE
# ==============================================================================
print("=" * 60)
print("1. STRUCTURE GENERALE")
print("=" * 60)
print(f"Lignes       : {len(df):,}")
print(f"Colonnes     : {df.shape[1]}")
print(f"Elections    : {sorted(df['id_election'].unique())}")
print(f"Communes     : {df['code_commune'].nunique()} uniques")
print(f"Types        : {df['type_election'].unique().tolist()}")

# ==============================================================================
# 2. VALEURS MANQUANTES
# ==============================================================================
print("\n" + "=" * 60)
print("2. VALEURS MANQUANTES")
print("=" * 60)
nan_counts = df.isna().sum()
nan_pct    = (nan_counts / len(df) * 100).round(1)
df_nan     = pd.DataFrame({"nb_nan": nan_counts, "pct": nan_pct})
df_nan     = df_nan[df_nan["nb_nan"] > 0].sort_values("pct", ascending=False)
print(df_nan.to_string())

# ==============================================================================
# 3. DISTRIBUTION DE LA VARIABLE CIBLE
# ==============================================================================
print("\n" + "=" * 60)
print("3. VARIABLE CIBLE — famille gagnante par election")
print("=" * 60)
for id_elec in sorted(df["id_election"].unique()):
    df_e   = df[df["id_election"] == id_elec]
    total  = len(df_e)
    counts = df_e["famille_regroupee"].value_counts()
    print(f"\n{id_elec} ({total} communes)")
    for famille, nb in counts.items():
        barre = "#" * int(nb / total * 40)
        print(f"  {famille:<20} {nb:>4} communes ({nb/total*100:.1f}%) {barre}")

# ==============================================================================
# 4. DESEQUILIBRE DES CLASSES (2024)
# ==============================================================================
print("\n" + "=" * 60)
print("4. DESEQUILIBRE DES CLASSES — Legislatives 2024")
print("=" * 60)
df_2024   = df[df["id_election"] == "2024_legi_t1"].copy()
total_24  = len(df_2024)
counts_24 = df_2024["famille_regroupee"].value_counts()
baseline  = counts_24.max() / total_24 * 100
for famille, nb in counts_24.items():
    print(f"  {famille:<20} {nb:>4} ({nb/total_24*100:.1f}%)")
print(f"\n  Baseline majority : {baseline:.1f}%")
print(f"  => Un modele doit depasser {baseline:.1f}% d'accuracy pour apporter de la valeur")

# ==============================================================================
# 5. STATISTIQUES SOCIOECO PAR FAMILLE (2024)
# ==============================================================================
print("\n" + "=" * 60)
print("5. STATISTIQUES SOCIOECO PAR FAMILLE — Legislatives 2024")
print("=" * 60)
cols_cles = ["revenu_median", "taux_chomage_2022", "taux_pauvrete",
             "nb_immigres", "population", "total_delits"]
cols_cles_dispo = [c for c in cols_cles if c in df_2024.columns]

for col in cols_cles_dispo:
    print(f"\n  {col}")
    print(f"  {'Famille':<20} {'Moyenne':>10} {'Mediane':>10} {'Min':>10} {'Max':>10}")
    print(f"  {'-'*62}")
    for famille in counts_24.index:
        vals = pd.to_numeric(
            df_2024[df_2024["famille_regroupee"] == famille][col], errors="coerce"
        ).dropna()
        if len(vals) > 0:
            print(f"  {famille:<20} {vals.mean():>10.1f} {vals.median():>10.1f} "
                  f"{vals.min():>10.1f} {vals.max():>10.1f}")

# ==============================================================================
# 6. SCORES EUROPEENNES PAR FAMILLE (2024)
# ==============================================================================
print("\n" + "=" * 60)
print("6. SCORES EUROPEENNES PAR FAMILLE — Legislatives 2024")
print("=" * 60)
cols_euro_dispo = [c for c in COLS_EURO if c in df_2024.columns]

for col in cols_euro_dispo:
    print(f"\n  {col.replace('euro_', '').replace('_ratio', '')}")
    for famille in counts_24.index:
        vals = pd.to_numeric(
            df_2024[df_2024["famille_regroupee"] == famille][col], errors="coerce"
        ).dropna()
        if len(vals) > 0:
            print(f"    {famille:<20} moy={vals.mean():>5.1f}%  med={vals.median():>5.1f}%")

# ==============================================================================
# 7. STABILITE TERRITORIALE 2017 -> 2024
# ==============================================================================
print("\n" + "=" * 60)
print("7. STABILITE TERRITORIALE — Legislatives 2017 -> 2024")
print("=" * 60)
df_legi = df[df["type_election"] == "legi"].copy()
df_legi["famille_regroupee"] = df_legi["famille_gagnante"].map(REGROUPEMENT).fillna("Divers")

df_pivot = df_legi.pivot_table(
    index="code_commune", columns="id_election",
    values="famille_regroupee", aggfunc="first"
)

if "2017_legi_t1" in df_pivot.columns and "2024_legi_t1" in df_pivot.columns:
    df_pivot = df_pivot.dropna(subset=["2017_legi_t1", "2024_legi_t1"])
    df_pivot["stable"] = df_pivot["2017_legi_t1"] == df_pivot["2024_legi_t1"]
    nb_stable   = df_pivot["stable"].sum()
    nb_instable = (~df_pivot["stable"]).sum()
    total_com   = len(df_pivot)

    print(f"\n  Communes analysees  : {total_com}")
    print(f"  Stables 2017->2024  : {nb_stable} ({nb_stable/total_com*100:.1f}%)")
    print(f"  Basculees           : {nb_instable} ({nb_instable/total_com*100:.1f}%)")

    print(f"\n  Flux de bascule :")
    df_bascule = df_pivot[~df_pivot["stable"]].copy()
    df_bascule["flux"] = df_bascule["2017_legi_t1"] + " -> " + df_bascule["2024_legi_t1"]
    for flux, nb in df_bascule["flux"].value_counts().items():
        barre = "#" * nb
        print(f"    {flux:<35} {nb:>3} communes {barre}")

# ==============================================================================
# 8. CORRELATIONS SOCIOECO (2024)
# ==============================================================================
print("\n" + "=" * 60)
print("8. CORRELATIONS ENTRE FEATURES SOCIOECO — Legislatives 2024")
print("=" * 60)
cols_socio_dispo = [c for c in COLS_SOCIO if c in df_2024.columns]
df_num = df_2024[cols_socio_dispo].apply(pd.to_numeric, errors="coerce")
corr   = df_num.corr()

print("\n  Paires les plus correlees (|r| > 0.7) :")
seuil = 0.7
paires = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > seuil:
            paires.append((corr.columns[i], corr.columns[j], round(r, 3)))

paires.sort(key=lambda x: abs(x[2]), reverse=True)
for c1, c2, r in paires:
    signe = "+" if r > 0 else "-"
    print(f"  {signe} {c1:<35} <-> {c2:<35} r={r:.3f}")

print("\nEDA terminee.")

1. STRUCTURE GENERALE
Lignes       : 1,360
Colonnes     : 41
Elections    : ['2017_legi_t1', '2017_pres_t1', '2022_legi_t1', '2022_pres_t1', '2024_legi_t1']
Communes     : 280 uniques
Types        : ['legi', 'pres']

2. VALEURS MANQUANTES
                                 nb_nan   pct
euro_droite_souverainiste_ratio     826  60.7
taux_pauvrete                       826  60.7
euro_ecologie_ratio                 560  41.2
nb_menages                          560  41.2
nb_personnes_menages                560  41.2
nb_logements_vacants                 35   2.6
population                           35   2.6
nb_logements                         35   2.6
nb_residences_principales            35   2.6
actifs_2022                          35   2.6
superficie_km2                       35   2.6
densite                              35   2.6
chomeurs_2022                        35   2.6
taux_chomage_2022                    35   2.6
total_delits                         35   2.6
inactifs_2022            